# DataCost Bing search analysis
Reproducible validation and calculations for the six Bing Webmaster Tools CSV exports supplied on 2026-08-13. The exports cover 2026-05-13 through 2026-08-11.

In [1]:
from pathlib import Path
import os
import pandas as pd

source_dir = Path(os.environ.get('DATACOST_BING_EXPORT_DIR', '.')).expanduser()
files = {
    'performance': 'datacost.co.za_SearchPerformanceOverview_All_8_13_2026.csv',
    'ai': 'datacost.co.za_AIPerformanceOverviewStats_8_13_2026.csv',
    'pages': 'datacost.co.za_PageTrafficReport_8_13_2026.csv',
    'keywords': 'datacost.co.za_KeywordReport_8_13_2026.csv',
    'countries': 'datacost.co.za_CountryReport_8_13_2026.csv',
    'devices': 'datacost.co.za_DeviceReport_8_13_2026.csv',
}
data = {name: pd.read_csv(source_dir / filename) for name, filename in files.items()}
{name: frame.shape for name, frame in data.items()}

{'performance': (91, 4),
 'ai': (91, 3),
 'pages': (55, 5),
 'keywords': (666, 5),
 'countries': (11, 5),
 'devices': (2, 5)}

In [2]:
performance = data['performance'].copy()
performance['Date'] = pd.to_datetime(performance['Date'], format='%m/%d/%Y %I:%M:%S %p')
latest_date = performance['Date'].max()
latest_28 = performance[performance['Date'].between(latest_date - pd.Timedelta(days=27), latest_date)]
previous_28 = performance[performance['Date'].between(latest_date - pd.Timedelta(days=55), latest_date - pd.Timedelta(days=28))]

def period_summary(frame):
    clicks = frame['Clicks'].sum()
    impressions = frame['Impressions'].sum()
    return {'clicks': int(clicks), 'impressions': int(impressions), 'ctr_pct': round(clicks / impressions * 100, 2)}

periods = pd.DataFrame([period_summary(latest_28), period_summary(previous_28)], index=['Latest 28 days', 'Previous 28 days'])
periods['click_change_pct'] = [round((periods.iloc[0].clicks / periods.iloc[1].clicks - 1) * 100, 1), None]
periods['impression_change_pct'] = [round((periods.iloc[0].impressions / periods.iloc[1].impressions - 1) * 100, 1), None]
periods

,clicks,impressions,ctr_pct,click_change_pct,impression_change_pct
Latest 28 days,307,22454,1.37,117.7,73.9
Previous 28 days,141,12913,1.09,NaN,NaN


In [3]:
totals = period_summary(performance)
reconciliation = pd.DataFrame([
    {'view': 'performance', 'clicks': totals['clicks'], 'impressions': totals['impressions']},
    {'view': 'countries', 'clicks': data['countries']['Clicks'].sum(), 'impressions': data['countries']['Impressions'].sum()},
    {'view': 'devices', 'clicks': data['devices']['Clicks'].sum(), 'impressions': data['devices']['Impressions'].sum()},
    {'view': 'pages', 'clicks': data['pages']['Clicks'].sum(), 'impressions': data['pages']['Impressions'].sum()},
    {'view': 'keywords', 'clicks': data['keywords']['Clicks'].sum(), 'impressions': data['keywords']['Impressions'].sum()},
])
reconciliation['click_coverage_pct'] = (reconciliation.clicks / totals['clicks'] * 100).round(1)
reconciliation['impression_coverage_pct'] = (reconciliation.impressions / totals['impressions'] * 100).round(1)
reconciliation

,view,clicks,impressions,click_coverage_pct,impression_coverage_pct
0,performance,671,47545,100.0,100.0
1,countries,671,47545,100.0,100.0
2,devices,671,47545,100.0,100.0
3,pages,606,44915,90.3,94.5
4,keywords,322,6024,48.0,12.7


In [4]:
devices = data['devices'].copy()
devices['click_share_pct'] = (devices.Clicks / devices.Clicks.sum() * 100).round(1)
devices['impression_share_pct'] = (devices.Impressions / devices.Impressions.sum() * 100).round(1)
devices

,Device,Impressions,Clicks,CTR,Avg. Position,click_share_pct,impression_share_pct
0,Mobile,16481,392,2.38%,4.66,58.4,34.7
1,Desktop,31064,279,0.9%,5.94,41.6,65.3


In [5]:
ai = data['ai'].copy()
ai['Date'] = pd.to_datetime(ai['Date'], format='%m/%d/%Y %I:%M:%S %p')
ai_latest = ai[ai['Date'].between(latest_date - pd.Timedelta(days=27), latest_date)]
ai_previous = ai[ai['Date'].between(latest_date - pd.Timedelta(days=55), latest_date - pd.Timedelta(days=28))]
ai_summary = pd.DataFrame([
    {'window': 'Latest 28 days', 'citations': ai_latest.Citations.sum(), 'avg_cited_pages': ai_latest['Cited Pages'].mean()},
    {'window': 'Previous 28 days', 'citations': ai_previous.Citations.sum(), 'avg_cited_pages': ai_previous['Cited Pages'].mean()},
])
ai_summary

,window,citations,avg_cited_pages
0,Latest 28 days,22365,21.714286
1,Previous 28 days,34590,12.178571


## Interpretation guardrails
Country and device totals reconcile to the performance export. Page rows are a high-coverage subset; keyword rows are materially incomplete. The `ww` country code and zero country positions are not decision-usable. AI citations are visibility counts, not search visits or conversions.